# Snapchat Activity Breakdown

## Problem Statement

You are given a log of social-media user activities and a table that maps users to age buckets.

For each age bucket, determine how members split their time between sending and opening content.

## Input Tables

### sb_activities

| Column Name | Data Type |
|------------|-----------|
| activity_id | INT |
| user_id | INT |
| activity_type | VARCHAR |
| time_spent | DECIMAL |
| activity_date | TIMESTAMP |

### sb_age_breakdown

| Column Name | Data Type |
|------------|-----------|
| user_id | INT |
| age_bucket | VARCHAR |

## Requirements

- Analyze activity information by age bucket.
- Consider send and open activities for percentage calculations.
- Ignore other activity types.
- If an age bucket has no send and no open activity time, the output percentages should be NULL.
- Sort results by `age_bucket` ascending.
- Return results matching the required output schema and order.

## Output Columns

| Column Name |
|------------|
| age_bucket |
| send_perc |
| open_perc |

## Sample Input

### sb_activities

| activity_id | user_id | activity_type | time_spent | activity_date |
|------------|---------|---------------|------------|---------------|
| 7274 | 123 | open | 4.5 | 2022-06-22 12:00:00 |
| 2425 | 123 | send | 3.5 | 2022-06-22 12:00:00 |
| 1413 | 456 | send | 5.67 | 2022-06-23 12:00:00 |
| 1414 | 789 | chat | 11.0 | 2022-06-25 12:00:00 |
| 2536 | 456 | open | 3.0 | 2022-06-25 12:00:00 |

### sb_age_breakdown

| user_id | age_bucket |
|---------|------------|
| 123 | 31-35 |
| 456 | 26-30 |
| 789 | 21-25 |

## Sample Output

| age_bucket | send_perc | open_perc |
|------------|-----------|-----------|
| 21-25 | NULL | NULL |
| 26-30 | 65.4 | 34.6 |
| 31-35 | 43.75 | 56.25 |

## Expected Output Schema

| Column Name | Data Type |
|------------|-----------|
| age_bucket | STRING |
| send_perc | DECIMAL |
| open_perc | DECIMAL |

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# sb_activities
sb_activities_schema = StructType([
    StructField("activity_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("activity_type", StringType(), True),
    StructField("time_spent", DoubleType(), True),
    StructField("activity_date", TimestampType(), True)
])

sb_activities_data = [
    (7274, 123, "open", 4.5, "2022-06-22 12:00:00"),
    (2425, 123, "send", 3.5, "2022-06-22 12:00:00"),
    (1413, 456, "send", 5.67, "2022-06-23 12:00:00"),
    (1414, 789, "chat", 11.0, "2022-06-25 12:00:00"),
    (2536, 456, "open", 3.0, "2022-06-25 12:00:00")
]

sb_activities_df = spark.createDataFrame(
    sb_activities_data,
    ["activity_id", "user_id", "activity_type", "time_spent", "activity_date"]
).withColumn(
    "activity_date",
    to_timestamp(col("activity_date"))
)

# sb_age_breakdown
sb_age_breakdown_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("age_bucket", StringType(), True)
])

sb_age_breakdown_data = [
    (123, "31-35"),
    (456, "26-30"),
    (789, "21-25")
]

sb_age_breakdown_df = spark.createDataFrame(
    sb_age_breakdown_data,
    schema=sb_age_breakdown_schema
)

In [0]:
result_df = (
    sb_activities_df.join(sb_age_breakdown_df, on="user_id", how="inner")
    .groupBy("age_bucket")
    .agg(
        sum("time_spent").alias("total_time_spent"),
        sum(when(col("activity_type") == "open", col("time_spent")).otherwise(0)).alias(
            "total_open_time"
        ),
        sum(when(col("activity_type") == "send", col("time_spent")).otherwise(0)).alias(
            "total_send_time"
        ),
    )
    .select(
        col("age_bucket"),
        when(round(col("total_send_time") * 100 / col("total_time_spent"), 2) == 0, lit(None))
        .otherwise(round(col("total_send_time") * 100 / col("total_time_spent"), 2))
        .alias("send_perc"),
        when(round(col("total_open_time") * 100 / col("total_time_spent"), 2) == 0, lit(None))
        .otherwise(round(col("total_open_time") * 100 / col("total_time_spent"), 2))
        .alias("open_perc"),
    )
    .orderBy(col("age_bucket"))
)
display(result_df)